# Arete Tech: High-Performance Logistics EDA Workbook
## 25 Polars LazyFrame Challenges | Global Supply Chain, Fleet Logistics & Inventory Management

**Assignee:** Yoksh Patil  
**Date:** June 3, 2026  
**Company:** Arete Tech  
**Complexity:** Industry-Grade

---

### Dataset Context
- **shipments_lazy**: Package movements (tracking_id, origin_warehouse_id, destination_zone, carrier_name, weight_kg, status, dispatched_timestamp, delivery_timestamp, route_code)
- **inventory_lazy**: Stock levels (sku_id, warehouse_id, quantity_on_hand, reorder_level, supplier_cost, last_audit_date)

In [1]:
import polars as pl
import polars.selectors as cs
from datetime import datetime, timedelta
import random
import numpy as np
import tempfile
import os
from pathlib import Path

print(f"Polars Version: {pl.__version__}")

Polars Version: 1.41.2


In [2]:
def generate_mock_datasets(n_shipments=1000, n_inventory=500):
    """
    Generate realistic mock data for supply chain analysis
    """
    random.seed(42)
    np.random.seed(42)
    
    # Warehouse and carrier data
    warehouses = ['WH-001', 'WH-002', 'WH-003', 'WH-004', 'WH-005']
    zones = ['US-WEST', 'US-CENTRAL', 'US-EAST', 'EU-NORTH', 'ASIA-PACIFIC']
    carriers = ['FedEx', 'UPS', 'DHL', 'DPD', None]  # Include None for testing
    statuses = ['Pending', 'Dispatched', 'In-Transit', 'Delivered', 'Delayed', 'Returned']
    skus = [f'SKU-{i:05d}' for i in range(1, n_inventory + 1)]
    
    # Generate Shipments
    base_date = datetime(2024, 1, 1)
    shipments_data = []
    
    for i in range(n_shipments):
        dispatch_date = base_date + timedelta(days=random.randint(0, 180))
        # Occasionally create chronological inversions (delivery before dispatch)
        days_in_transit = random.randint(-5, 15) if random.random() < 0.05 else random.randint(1, 15)
        delivery_date = dispatch_date + timedelta(days=days_in_transit)
        
        route_codes = ['US-ORD-ZONE1', 'US-LAX-ZONE2', 'EU-CDG-ZONE3', 'ASIA-HND-ZONE4', 'US-ATL-ZONE5']
        
        shipments_data.append({
            'tracking_id': f'TRK-{i:08d}',
            'origin_warehouse_id': random.choice(warehouses),
            'destination_zone': random.choice(zones),
            'carrier_name': random.choice(carriers),
            'weight_kg': round(random.uniform(10, 2000), 2),
            'status': random.choice(statuses),
            'dispatched_timestamp': dispatch_date.strftime('%Y-%m-%d %H:%M:%S'),
            'delivery_timestamp': delivery_date.strftime('%Y-%m-%d %H:%M:%S'),
            'route_code': random.choice(route_codes),
            'driver_notes': f'Note_{i}' * random.randint(1, 5),  # Large text field
            'manifest_comments': f'Comment_{i}' * random.randint(1, 5)  # Large text field
        })
    
    # Generate Inventory
    inventory_data = []
    for i, sku in enumerate(skus):
        inventory_data.append({
            'sku_id': sku,
            'warehouse_id': random.choice(warehouses),
            'quantity_on_hand': random.randint(0, 10000) if random.random() > 0.1 else None,
            'reorder_level': random.randint(100, 1000),
            'supplier_cost': round(random.uniform(5, 500), 2),
            'last_audit_date': (base_date + timedelta(days=random.randint(0, 180))).strftime('%Y-%m-%d')
        })
    
    shipments_df = pl.DataFrame(shipments_data)
    inventory_df = pl.DataFrame(inventory_data)
    
    return shipments_df, inventory_df

# Generate datasets
print("Generating mock datasets...")
shipments_df, inventory_df = generate_mock_datasets(n_shipments=1000, n_inventory=300)

# Create lazy frames
shipments_lazy = shipments_df.lazy()
inventory_lazy = inventory_df.lazy()

print(f"✓ Shipments LazyFrame: {shipments_df.shape[0]} rows")
print(f"✓ Inventory LazyFrame: {inventory_df.shape[0]} rows")

Generating mock datasets...
✓ Shipments LazyFrame: 1000 rows
✓ Inventory LazyFrame: 300 rows


In [3]:
# Challenge 1: Zero-Memory Schema Inspection
# Scan without materializing any rows - extract schema from lazy frame

print("\n" + "="*80)
print("CHALLENGE 1: Zero-Memory Schema Inspection")
print("="*80)

# Get schema WITHOUT materializing any rows
schema = shipments_lazy.collect_schema()
print("\nShipments LazyFrame Schema (Zero-Materialization):")
for col_name, data_type in schema.items():
    print(f"  {col_name:30s} -> {data_type}")

print("\nInventory LazyFrame Schema:")
schema_inv = inventory_lazy.collect_schema()
for col_name, data_type in schema_inv.items():
    print(f"  {col_name:30s} -> {data_type}")


CHALLENGE 1: Zero-Memory Schema Inspection

Shipments LazyFrame Schema (Zero-Materialization):
  tracking_id                    -> String
  origin_warehouse_id            -> String
  destination_zone               -> String
  carrier_name                   -> String
  weight_kg                      -> Float64
  status                         -> String
  dispatched_timestamp           -> String
  delivery_timestamp             -> String
  route_code                     -> String
  driver_notes                   -> String
  manifest_comments              -> String

Inventory LazyFrame Schema:
  sku_id                         -> String
  warehouse_id                   -> String
  quantity_on_hand               -> Int64
  reorder_level                  -> Int64
  supplier_cost                  -> Float64
  last_audit_date                -> String


In [4]:
print("\n" + "="*80)
print("CHALLENGE 2: ISO Datetime Standardization")
print("="*80)

# Convert string timestamps to microsecond-precision pl.Datetime within lazy query plan
result_c2 = (
    shipments_lazy
    .with_columns([
        pl.col('dispatched_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').cast(pl.Datetime('us')).alias('dispatch_dt_us'),
        pl.col('delivery_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').cast(pl.Datetime('us')).alias('delivery_dt_us')
    ])
    .select(['tracking_id', 'dispatch_dt_us', 'delivery_dt_us'])
    .limit(5)
    .collect()
)

print("\nDatetime Standardization Results:")
print(result_c2)
print(f"Data types: {result_c2.schema}")


CHALLENGE 2: ISO Datetime Standardization

Datetime Standardization Results:
shape: (5, 3)
┌──────────────┬─────────────────────┬─────────────────────┐
│ tracking_id  ┆ dispatch_dt_us      ┆ delivery_dt_us      │
│ ---          ┆ ---                 ┆ ---                 │
│ str          ┆ datetime[μs]        ┆ datetime[μs]        │
╞══════════════╪═════════════════════╪═════════════════════╡
│ TRK-00000000 ┆ 2024-06-12 00:00:00 ┆ 2024-06-24 00:00:00 │
│ TRK-00000001 ┆ 2024-04-18 00:00:00 ┆ 2024-04-15 00:00:00 │
│ TRK-00000002 ┆ 2024-02-26 00:00:00 ┆ 2024-03-02 00:00:00 │
│ TRK-00000003 ┆ 2024-01-24 00:00:00 ┆ 2024-01-30 00:00:00 │
│ TRK-00000004 ┆ 2024-04-06 00:00:00 ┆ 2024-04-11 00:00:00 │
└──────────────┴─────────────────────┴─────────────────────┘
Data types: Schema([('tracking_id', String), ('dispatch_dt_us', Datetime(time_unit='us', time_zone=None)), ('delivery_dt_us', Datetime(time_unit='us', time_zone=None))])


In [ ]:
print("\n" + "="*80)
print("CHALLENGE 3: Structural Column Eviction")
print("="*80)

# Drop large text columns during lazy ingestion using Polars selectors
result_c3 = (
    shipments_lazy
    .select(~cs.by_name(['driver_notes', 'manifest_comments']))  # Exclude large text fields
    .limit(3)
    .collect()
)

print("\nColumns remaining after eviction:")
print(f"Original columns: {shipments_lazy.collect_schema().keys()}")
print(f"After eviction: {result_c3.columns}")
print(f"\nMemory saved by dropping text columns")

# Challenge 4: Chronological Inversion Identification
print("\n" + "="*80)
print("CHALLENGE 4: Chronological Inversion Identification")
print("="*80)

result_c4 = (
    shipments_lazy
    .with_columns([
        pl.col('dispatched_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('dispatch_dt'),
        pl.col('delivery_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('delivery_dt')
    ])
    .filter(pl.col('delivery_dt') < pl.col('dispatch_dt'))  # Chronological inversions
    .select(['tracking_id', 'dispatch_dt', 'delivery_dt'])
    .collect()
)

print(f"\nMalformed records detected: {result_c4.shape[0]}")
if result_c4.shape[0] > 0:
    print(result_c4.head())

# Challenge 5: Type-Safe Null Schema Hardening
print("\n" + "="*80)
print("CHALLENGE 5: Type-Safe Null Schema Hardening")
print("="*80)

# Enforce strict schema during lazy scan
strict_schema = {
    'sku_id': pl.Utf8,
    'warehouse_id': pl.Utf8,
    'quantity_on_hand': pl.Int64,  # Nulls will be pl.Int64, not strings
    'reorder_level': pl.Int64,
    'supplier_cost': pl.Float64,
    'last_audit_date': pl.Utf8
}

result_c5 = (
    inventory_lazy
    .select(pl.col('*').cast(strict_schema))
    .filter(pl.col('quantity_on_hand').is_null())
    .select(['sku_id', 'quantity_on_hand'])
    .collect()
)

print(f"\nSKUs with null quantity_on_hand (enforced as Int64): {result_c5.shape[0]}")
print(f"Data type of quantity_on_hand: {result_c5.schema['quantity_on_hand']}")
print(result_c5.head())

## CATEGORY 1 & 2: Schema Validation & Filtering

In [7]:
print("\n" + "="*80)
print("CHALLENGE 6: Critical Logistics Bottleneck Slicing")
print("="*80)

result_c6 = (
    shipments_lazy
    .filter(
        (pl.col('status') == 'Delayed') &
        (pl.col('weight_kg') > 500) &
        (pl.col('carrier_name').is_not_null())
    )
    .select(['tracking_id', 'status', 'weight_kg', 'carrier_name'])
    .collect()
)

print(f"\nCritical bottleneck shipments: {result_c6.shape[0]}")
print(result_c6.head())

# Challenge 7: Clean Categorical Normalization
print("\n" + "="*80)
print("CHALLENGE 7: Clean Categorical Normalization")
print("="*80)

result_c7 = (
    shipments_lazy
    .with_columns(
        pl.col('carrier_name')
        .fill_null('independent_contractor')
        .str.to_lowercase()
        .str.strip_chars()
        .alias('carrier_normalized')
    )
    .select(['tracking_id', 'carrier_normalized'])
    .collect()
)

print(f"\nNormalized carriers:")
print(result_c7['carrier_normalized'].unique().sort())

# Challenge 8: Native Regex Routing Code Extraction
print("\n" + "="*80)
print("CHALLENGE 8: Native Regex Routing Code Extraction")
print("="*80)

result_c8 = (
    shipments_lazy
    .with_columns(
        pl.col('route_code')
        .str.extract(r'US-([A-Z]{3})', 1)
        .alias('regional_hub')
    )
    .select(['tracking_id', 'route_code', 'regional_hub'])
    .collect()
)

print(f"\nExtracted regional hubs:")
print(result_c8.head())


CHALLENGE 6: Critical Logistics Bottleneck Slicing

Critical bottleneck shipments: 97
shape: (5, 4)
┌──────────────┬─────────┬───────────┬──────────────┐
│ tracking_id  ┆ status  ┆ weight_kg ┆ carrier_name │
│ ---          ┆ ---     ┆ ---       ┆ ---          │
│ str          ┆ str     ┆ f64       ┆ str          │
╞══════════════╪═════════╪═══════════╪══════════════╡
│ TRK-00000019 ┆ Delayed ┆ 1449.47   ┆ UPS          │
│ TRK-00000041 ┆ Delayed ┆ 1216.36   ┆ DPD          │
│ TRK-00000048 ┆ Delayed ┆ 1773.41   ┆ DPD          │
│ TRK-00000068 ┆ Delayed ┆ 1116.02   ┆ DHL          │
│ TRK-00000089 ┆ Delayed ┆ 1106.17   ┆ DHL          │
└──────────────┴─────────┴───────────┴──────────────┘

CHALLENGE 7: Clean Categorical Normalization

Normalized carriers:
shape: (5,)
Series: 'carrier_normalized' [str]
[
	"dhl"
	"dpd"
	"fedex"
	"independent_contractor"
	"ups"
]

CHALLENGE 8: Native Regex Routing Code Extraction

Extracted regional hubs:
shape: (5, 3)
┌──────────────┬────────────────┬──────

In [ ]:
print("\n" + "="*80)
print("CHALLENGE 9: Systemic Risk Flagging")
print("="*80)

result_c9 = (
    shipments_lazy
    .with_columns([
        pl.col('dispatched_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('dispatch_dt'),
        pl.col('delivery_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('delivery_dt'),
    ])
    .with_columns(
        transit_hours = (pl.col('delivery_dt') - pl.col('dispatch_dt')).dt.total_hours()
    )
    .with_columns(
        pl.when(
            (pl.col('weight_kg') > 750) &
            (pl.col('status') == 'In-Transit') &
            (pl.col('transit_hours') > 72)
        )
        .then(True)
        .otherwise(False)
        .alias('is_high_risk_transit')
    )
    .filter(pl.col('is_high_risk_transit'))
    .select(['tracking_id', 'weight_kg', 'status', 'transit_hours', 'is_high_risk_transit'])
    .collect()
)

print(f"\nHigh-risk shipments detected: {result_c9.shape[0]}")
if result_c9.shape[0] > 0:
    print(result_c9.head())

# Challenge 10: Financial Asset Cost Tiering
print("\n" + "="*80)
print("CHALLENGE 10: Financial Asset Cost Tiering")
print("="*80)

result_c10 = (
    inventory_lazy
    .with_columns(
        pl.when(pl.col('supplier_cost') < 20)
        .then(pl.lit('Economy'))
        .when((pl.col('supplier_cost') >= 20) & (pl.col('supplier_cost') <= 100))
        .then(pl.lit('Standard'))
        .otherwise(pl.lit('Premium'))
        .alias('asset_value_tier')
    )
    .select(['sku_id', 'supplier_cost', 'asset_value_tier'])
    .collect()
)

print(f"\nInventory tiering distribution:")
print(result_c10['asset_value_tier'].value_counts().sort('counts', descending=True))
print(f"\nSample records:")
print(result_c10.head())

## CATEGORY 3, 4 & 5: Grouping, Windows & Joins

In [6]:
print("\n" + "="*80)
print("CHALLENGE 11: Consolidated Warehouse Scorecard")
print("="*80)

result_c11 = (
    inventory_lazy
    .group_by('warehouse_id')
    .agg([
        (pl.col('quantity_on_hand') * pl.col('supplier_cost')).sum().alias('total_capital_tied'),
        pl.col('quantity_on_hand').mean().alias('avg_stock_volume'),
        pl.col('sku_id').n_unique().alias('unique_skus')
    ])
    .sort('total_capital_tied', descending=True)
    .collect()
)

print("\nWarehouse Scorecard:")
print(result_c11)

# Challenge 12: Carrier Efficiency Ranking
print("\n" + "="*80)
print("CHALLENGE 12: Carrier Efficiency Ranking")
print("="*80)

result_c12 = (
    shipments_lazy
    .with_columns([
        pl.col('dispatched_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('dispatch_dt'),
        pl.col('delivery_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('delivery_dt'),
    ])
    .filter(pl.col('carrier_name').is_not_null())
    .with_columns(
        transit_hours = (pl.col('delivery_dt') - pl.col('dispatch_dt')).dt.total_hours()
    )
    .group_by('carrier_name')
    .agg(pl.col('transit_hours').mean().alias('avg_transit_hours'))
    .sort('avg_transit_hours')
    .limit(5)
    .collect()
)

print("\nTop 5 Most Efficient Carriers:")
print(result_c12)

# Challenge 13: Inventory Auditing Lifespan
print("\n" + "="*80)
print("CHALLENGE 13: Inventory Auditing Lifespan")
print("="*80)

result_c13 = (
    inventory_lazy
    .group_by('sku_id')
    .agg([
        pl.col('last_audit_date').min().alias('earliest_audit'),
        pl.col('last_audit_date').max().alias('latest_audit')
    ])
    .with_columns(
        audit_lifespan_days = (
            pl.col('latest_audit').str.to_date() - 
            pl.col('earliest_audit').str.to_date()
        ).dt.total_days()
    )
    .select(['sku_id', 'earliest_audit', 'latest_audit', 'audit_lifespan_days'])
    .collect()
)

print(f"\nInventory audit lifespan statistics:")
print(result_c13.describe())


CHALLENGE 11: Consolidated Warehouse Scorecard

Warehouse Scorecard:
shape: (5, 4)
┌──────────────┬────────────────────┬──────────────────┬─────────────┐
│ warehouse_id ┆ total_capital_tied ┆ avg_stock_volume ┆ unique_skus │
│ ---          ┆ ---                ┆ ---              ┆ ---         │
│ str          ┆ f64                ┆ f64              ┆ u32         │
╞══════════════╪════════════════════╪══════════════════╪═════════════╡
│ WH-002       ┆ 6.5539e7           ┆ 5591.930233      ┆ 51          │
│ WH-003       ┆ 6.5191e7           ┆ 5307.396226      ┆ 62          │
│ WH-001       ┆ 6.5052e7           ┆ 4714.537037      ┆ 63          │
│ WH-004       ┆ 6.1014e7           ┆ 4745.383333      ┆ 67          │
│ WH-005       ┆ 5.5108e7           ┆ 5213.14          ┆ 57          │
└──────────────┴────────────────────┴──────────────────┴─────────────┘

CHALLENGE 12: Carrier Efficiency Ranking

Top 5 Most Efficient Carriers:
shape: (4, 2)
┌──────────────┬───────────────────┐
│ carrier_

In [8]:
print("\n" + "="*80)
print("CHALLENGE 14: Non-Breaking Cumulative Volume Tracking")
print("="*80)

result_c14 = (
    shipments_lazy
    .with_columns(
        pl.col('dispatched_timestamp').str.to_datetime('%Y-%m-%d %H:%M:%S').alias('dispatch_dt')
    )
    .with_columns(
        dispatch_date = pl.col('dispatch_dt').dt.date()
    )
    .group_by('dispatch_date')
    .agg(pl.col('weight_kg').sum().alias('daily_weight'))
    .sort('dispatch_date')
    .with_columns(
        cumulative_weight = pl.col('daily_weight').cum_sum()
    )
    .collect()
)

print("\nDaily Cumulative Volume Tracking (first 10 days):")
print(result_c14.head(10))

# Challenge 15: Warehouse Stockout Rate Analysis
print("\n" + "="*80)
print("CHALLENGE 15: Warehouse Stockout Rate Analysis")
print("="*80)

result_c15 = (
    inventory_lazy
    .group_by('warehouse_id')
    .agg([
        pl.when(pl.col('quantity_on_hand') <= pl.col('reorder_level'))
        .then(1)
        .otherwise(0)
        .sum()
        .alias('stockout_items'),
        pl.col('sku_id').count().alias('total_items')
    ])
    .with_columns(
        stockout_rate_pct = (pl.col('stockout_items') / pl.col('total_items') * 100).round(2)
    )
    .sort('stockout_rate_pct', descending=True)
    .collect()
)

print("\nWarehouse Stockout Rates:")
print(result_c15)


CHALLENGE 14: Non-Breaking Cumulative Volume Tracking

Daily Cumulative Volume Tracking (first 10 days):
shape: (10, 3)
┌───────────────┬──────────────┬───────────────────┐
│ dispatch_date ┆ daily_weight ┆ cumulative_weight │
│ ---           ┆ ---          ┆ ---               │
│ date          ┆ f64          ┆ f64               │
╞═══════════════╪══════════════╪═══════════════════╡
│ 2024-01-01    ┆ 7345.25      ┆ 7345.25           │
│ 2024-01-02    ┆ 10922.75     ┆ 18268.0           │
│ 2024-01-03    ┆ 1550.08      ┆ 19818.08          │
│ 2024-01-04    ┆ 6353.26      ┆ 26171.34          │
│ 2024-01-05    ┆ 3654.23      ┆ 29825.57          │
│ 2024-01-06    ┆ 6560.92      ┆ 36386.49          │
│ 2024-01-07    ┆ 9188.09      ┆ 45574.58          │
│ 2024-01-08    ┆ 7199.99      ┆ 52774.57          │
│ 2024-01-09    ┆ 3777.02      ┆ 56551.59          │
│ 2024-01-10    ┆ 6864.08      ┆ 63415.67          │
└───────────────┴──────────────┴───────────────────┘

CHALLENGE 15: Warehouse Stocko

In [9]:
print("\n" + "="*80)
print("CHALLENGE 16: Local Checkpoint Sequence Indexing")
print("="*80)

# Generate checkpoint data to simulate warehouse checkpoints
checkpoint_data = []
for _ in range(50):
    tracking_id = f'TRK-{random.randint(1, 10):08d}'
    for cp_num in range(random.randint(1, 5)):
        checkpoint_data.append({
            'tracking_id': tracking_id,
            'checkpoint_id': f'WH-{random.randint(1, 5):03d}',
            'timestamp': (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))).isoformat()
        })

checkpoints_df = pl.DataFrame(checkpoint_data).sort(['tracking_id', 'timestamp'])
checkpoints_lazy = checkpoints_df.lazy()

result_c16 = (
    checkpoints_lazy
    .with_columns(
        checkpoint_seq = pl.col('tracking_id').cum_count().over('tracking_id')
    )
    .collect()
)

print("Checkpoint Sequence Indexing:")
print(result_c16.filter(pl.col('tracking_id') == result_c16['tracking_id'][0]).head(10))

# Challenge 17: Checkpoint Lag Mapping
print("\n" + "="*80)
print("CHALLENGE 17: Checkpoint Lag Mapping")
print("="*80)

result_c17 = (
    checkpoints_lazy
    .with_columns(
        previous_checkpoint = pl.col('checkpoint_id').shift(1).over('tracking_id')
    )
    .select(['tracking_id', 'checkpoint_id', 'previous_checkpoint'])
    .collect()
)

print("Preceding checkpoint location:")
print(result_c17.filter(pl.col('tracking_id') == result_c17['tracking_id'][0]).head(10))

# Challenge 18: Scan Velocity Fraud Detection
print("\n" + "="*80)
print("CHALLENGE 18: Scan Velocity Fraud Detection")
print("="*80)

result_c18 = (
    checkpoints_lazy
    .with_columns(
        pl.col('timestamp').str.to_datetime().alias('ts_dt')
    )
    .with_columns(
        time_diff_seconds = (pl.col('ts_dt').shift(-1).over('tracking_id') - pl.col('ts_dt')).dt.total_seconds()
    )
    .filter(
        (pl.col('time_diff_seconds') < 30) & 
        (pl.col('time_diff_seconds') > 0) &
        (pl.col('checkpoint_id').shift(-1).over('tracking_id') != pl.col('checkpoint_id'))
    )
    .select(['tracking_id', 'checkpoint_id', 'time_diff_seconds'])
    .collect()
)

print(f"Fraud detection anomalies (< 30 seconds between locations): {result_c18.shape[0]}")
if result_c18.shape[0] > 0:
    print(result_c18.head())


CHALLENGE 16: Local Checkpoint Sequence Indexing
Checkpoint Sequence Indexing:
shape: (10, 4)
┌──────────────┬───────────────┬─────────────────────┬────────────────┐
│ tracking_id  ┆ checkpoint_id ┆ timestamp           ┆ checkpoint_seq │
│ ---          ┆ ---           ┆ ---                 ┆ ---            │
│ str          ┆ str           ┆ str                 ┆ u32            │
╞══════════════╪═══════════════╪═════════════════════╪════════════════╡
│ TRK-00000001 ┆ WH-005        ┆ 2024-01-07T12:00:00 ┆ 1              │
│ TRK-00000001 ┆ WH-002        ┆ 2024-01-09T08:00:00 ┆ 2              │
│ TRK-00000001 ┆ WH-004        ┆ 2024-01-11T05:00:00 ┆ 3              │
│ TRK-00000001 ┆ WH-003        ┆ 2024-01-14T08:00:00 ┆ 4              │
│ TRK-00000001 ┆ WH-005        ┆ 2024-01-21T08:00:00 ┆ 5              │
│ TRK-00000001 ┆ WH-002        ┆ 2024-02-05T18:00:00 ┆ 6              │
│ TRK-00000001 ┆ WH-005        ┆ 2024-02-12T17:00:00 ┆ 7              │
│ TRK-00000001 ┆ WH-004        ┆ 2024-02-

### Challenge 19: Initial Intake Manifest Isolation

In [ ]:
print("\n" + "="*80)
print("CHALLENGE 19: Initial Intake Manifest Isolation")
print("="*80)

result_c19 = (
    checkpoints_lazy
    .with_columns(
        pl.col('timestamp').str.to_datetime().alias('ts_dt')
    )
    .with_columns(
        checkpoint_seq = pl.col('tracking_id').cum_count().over('tracking_id')
    )
    .filter(pl.col('checkpoint_seq') == 1)  # First check-in only
    .select(['tracking_id', 'checkpoint_id', 'ts_dt'])
    .collect()
)

print(f"First intake manifests extracted: {result_c19.shape[0]}")
print(result_c19.head())

# Challenge 20: Localized Inventory Valuation Contribution
print("\n" + "="*80)
print("CHALLENGE 20: Localized Inventory Valuation Contribution")
print("="*80)

result_c20 = (
    inventory_lazy
    .with_columns(
        sku_value = pl.col('quantity_on_hand') * pl.col('supplier_cost')
    )
    .with_columns(
        warehouse_total_value = pl.col('sku_value').sum().over('warehouse_id')
    )
    .with_columns(
        sku_pct_contribution = (
            (pl.col('sku_value') / pl.col('warehouse_total_value') * 100).round(2)
        )
    )
    .select(['sku_id', 'warehouse_id', 'sku_value', 'warehouse_total_value', 'sku_pct_contribution'])
    .collect()
)

print("SKU Valuation Contribution by Warehouse:")
print(result_c20.head())

## CATEGORY 5: Joining, Pivoting & Reshaping

### Challenge 21: Schema-Safe Operational Join

In [13]:
print("\n" + "="*80)
print("CHALLENGE 21: Schema-Safe Operational Join")
print("="*80)

# Prepare shipments for join - aggregate by warehouse
shipments_agg = (
    shipments_lazy
    .group_by('origin_warehouse_id')
    .agg(pl.col('weight_kg').sum().alias('total_weight_shipped'))
)

# Inner join with inventory on warehouse_id
result_c21 = (
    shipments_agg
    .join(
        inventory_lazy,
        left_on='origin_warehouse_id',
        right_on='warehouse_id',
        how='inner'
    )
    .select([
        pl.col('origin_warehouse_id').alias('warehouse'),
        pl.col('total_weight_shipped'),
        pl.col('sku_id'),
        pl.col('quantity_on_hand')
    ])
    .collect()
)

print("Schema-Safe Join Results:")
print(result_c21.head())

# Challenge 22: Cross-Zone Carrier Volume Pivoting
print("\n" + "="*80)
print("CHALLENGE 22: Cross-Zone Carrier Volume Pivoting")
print("="*80)

result_c22 = (
    shipments_lazy
    .filter(pl.col('carrier_name').is_not_null())
    .group_by(['carrier_name', 'destination_zone'])
    .agg(pl.col('weight_kg').sum().alias('weight'))
    .collect()
    .pivot(on='destination_zone', index='carrier_name', values='weight', aggregate_function='sum')
)

print("Carrier Volume by Destination Zone:")
print(result_c22)

# Challenge 23: Orphaned Item Audit (Anti-Join)
print("\n" + "="*80)
print("CHALLENGE 23: Orphaned Item Audit (Anti-Join)")
print("="*80)

# Create synthetic shipment manifest
shipment_skus = shipments_df.select('tracking_id').sample(100, seed=42)
shipment_skus_lazy = shipment_skus.lazy()

result_c23 = (
    inventory_lazy
    .join(
        shipment_skus_lazy,
        left_on='sku_id',
        right_on='tracking_id',
        how='anti'
    )
    .select(['sku_id', 'warehouse_id', 'quantity_on_hand'])
    .collect()
)

print(f"Orphaned SKUs (never in shipments): {result_c23.shape[0]}")
print(result_c23.head())


CHALLENGE 21: Schema-Safe Operational Join
Schema-Safe Join Results:
shape: (5, 4)
┌───────────┬──────────────────────┬───────────┬──────────────────┐
│ warehouse ┆ total_weight_shipped ┆ sku_id    ┆ quantity_on_hand │
│ ---       ┆ ---                  ┆ ---       ┆ ---              │
│ str       ┆ f64                  ┆ str       ┆ i64              │
╞═══════════╪══════════════════════╪═══════════╪══════════════════╡
│ WH-003    ┆ 195265.5             ┆ SKU-00001 ┆ null             │
│ WH-002    ┆ 183040.94            ┆ SKU-00002 ┆ 4831             │
│ WH-001    ┆ 193018.65            ┆ SKU-00003 ┆ 3592             │
│ WH-003    ┆ 195265.5             ┆ SKU-00004 ┆ 8804             │
│ WH-001    ┆ 193018.65            ┆ SKU-00005 ┆ 6258             │
└───────────┴──────────────────────┴───────────┴──────────────────┘

CHALLENGE 22: Cross-Zone Carrier Volume Pivoting
Carrier Volume by Destination Zone:
shape: (4, 6)
┌──────────────┬──────────┬──────────┬──────────────┬────────────┬──

### Challenge 24: Wide Stock Level Matrix Unpivoting

In [16]:
print("\n" + "="*80)
print("CHALLENGE 24: Wide Stock Level Matrix Unpivoting")
print("="*80)

# Create a wide-format quarterly inventory matrix
wide_data = []
quarters = ['q1_stock', 'q2_stock', 'q3_stock', 'q4_stock']
for i, sku in enumerate(['SKU-00001', 'SKU-00002', 'SKU-00003']):
    for q_idx, quarter in enumerate(quarters):
        wide_data.append({
            'sku_id': sku,
            'warehouse': 'WH-001',
            'q1_stock': random.randint(100, 500) if q_idx == 0 else None,
            'q2_stock': random.randint(100, 500) if q_idx == 1 else None,
            'q3_stock': random.randint(100, 500) if q_idx == 2 else None,
            'q4_stock': random.randint(100, 500) if q_idx == 3 else None
        })

wide_df = pl.DataFrame({
    'sku_id': ['SKU-00001', 'SKU-00002', 'SKU-00003'],
    'q1_stock': [250, 320, 410],
    'q2_stock': [280, 350, 420],
    'q3_stock': [310, 380, 450],
    'q4_stock': [340, 410, 480]
})

wide_lazy = wide_df.lazy()

# Unpivot to long format
result_c24 = (
    wide_lazy
    .unpivot(index='sku_id', on=quarters, variable_name='quarter', value_name='stock_level')
    .collect()
)

print("Unpivoted Quarterly Stock Levels:")
print(result_c24)

# Challenge 25: Safety Protocol Cross-Joining
print("\n" + "="*80)
print("CHALLENGE 25: Safety Protocol Cross-Joining")
print("="*80)

# Reference compliance check parameters
compliance_checks = pl.DataFrame({
    'check_id': ['CHK-001', 'CHK-002'],
    'check_name': ['Weight Limit', 'Hazmat Compliance'],
    'frequency': ['Daily', 'Weekly']
}).lazy()

# Hazardous warehouses
hazardous_warehouses = inventory_lazy.select('warehouse_id').unique().limit(2)

# Cross join
result_c25 = (
    compliance_checks
    .join(
        hazardous_warehouses,
        how='cross'
    )
    .collect()
)

print(f"Daily Inspection Grid Created: {result_c25.shape[0]} inspection tasks")
print(result_c25)


CHALLENGE 24: Wide Stock Level Matrix Unpivoting
Unpivoted Quarterly Stock Levels:
shape: (12, 3)
┌───────────┬──────────┬─────────────┐
│ sku_id    ┆ quarter  ┆ stock_level │
│ ---       ┆ ---      ┆ ---         │
│ str       ┆ str      ┆ i64         │
╞═══════════╪══════════╪═════════════╡
│ SKU-00001 ┆ q1_stock ┆ 250         │
│ SKU-00002 ┆ q1_stock ┆ 320         │
│ SKU-00003 ┆ q1_stock ┆ 410         │
│ SKU-00001 ┆ q2_stock ┆ 280         │
│ SKU-00002 ┆ q2_stock ┆ 350         │
│ …         ┆ …        ┆ …           │
│ SKU-00002 ┆ q3_stock ┆ 380         │
│ SKU-00003 ┆ q3_stock ┆ 450         │
│ SKU-00001 ┆ q4_stock ┆ 340         │
│ SKU-00002 ┆ q4_stock ┆ 410         │
│ SKU-00003 ┆ q4_stock ┆ 480         │
└───────────┴──────────┴─────────────┘

CHALLENGE 25: Safety Protocol Cross-Joining
Daily Inspection Grid Created: 4 inspection tasks
shape: (4, 4)
┌──────────┬───────────────────┬───────────┬──────────────┐
│ check_id ┆ check_name        ┆ frequency ┆ warehouse_id │
│ ---    